# Tennessee Eastman Process (TEP) — Fault Dataset Exploration

**Dataset**: 52 process variables (41 measured + 11 manipulated), 21 fault types, sampled every 3 minutes.  
**Files**: FaultFree Training/Testing + Faulty Training/Testing (read directly from zip).

In [ ]:
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = "../datasets/TEP/Harvard"
FF_TRAIN = f"{DATA_DIR}/ff_train.parquet"
F_TRAIN  = f"{DATA_DIR}/f_train.parquet"
FF_TEST  = f"{DATA_DIR}/ff_test.parquet"
F_TEST   = f"{DATA_DIR}/f_test.parquet"

XMEAS    = [f"xmeas_{i}" for i in range(1, 42)]
XMV      = [f"xmv_{i}"   for i in range(1, 12)]
FEATURES = XMEAS + XMV

FAULT_INJECTION_SAMPLE = 160
FAULT_NUMBERS = [1, 4, 5]
COMPARE_VARS  = ["xmeas_1", "xmeas_7", "xmeas_9", "xmeas_11", "xmv_3", "xmv_4"]
TRAJ_VARS     = ["xmeas_1", "xmeas_4", "xmeas_7", "xmeas_11"]
TRAJ_FAULTS   = [1, 2, 5, 11]

print("Libraries loaded.")

## 1. Load Data

In [ ]:
print("Loading ff_train...")
ff_train = pd.read_parquet(FF_TRAIN)
print(f"  ff_train : {ff_train.shape[0]:>8,} rows × {ff_train.shape[1]} cols  "
      f"({ff_train.memory_usage(deep=True).sum() / 1e6:.0f} MB)")

In [ ]:
# fault_counts: read only 2 columns — avoids loading 1+ GB
print("Extracting fault counts from f_train (2-col read)...")
fault_counts = (
    pd.read_parquet(F_TRAIN, columns=["faultNumber", "simulationRun"])
    .groupby("faultNumber")["simulationRun"].nunique()
    .reset_index(name="runs")
)

# Normalisation stats derived from the small ff_train
ff_mean = ff_train[FEATURES].mean()
ff_std  = ff_train[FEATURES].std().replace(0, 1)

# Process f_train one fault at a time: ~30 MB peak per iteration instead of ~1 GB all-at-once.
# Each chunk is post-injection rows for a single fault (filters push down to row-group level).
print("Processing f_train fault-by-fault (post-injection only)...")
dev_rows      = {}
sample_chunks = []
for fnum in range(1, 22):
    chunk = pd.read_parquet(
        F_TRAIN,
        columns=["faultNumber", "sample"] + FEATURES,
        filters=[("faultNumber", "==", fnum), ("sample", ">", FAULT_INJECTION_SAMPLE)],
    )
    chunk = chunk[(chunk["faultNumber"] == fnum) & (chunk["sample"] > FAULT_INJECTION_SAMPLE)]
    sample_chunks.append(chunk[FEATURES].sample(min(250, len(chunk)), random_state=fnum))
    dev_rows[fnum] = ((chunk[FEATURES] - ff_mean) / ff_std).abs().mean()
    del chunk

faulty_post_sample = (
    pd.concat(sample_chunks, ignore_index=True).sample(5000, random_state=42)
)
faulty_post_sample["label"] = "Faulty"
del sample_chunks

dev = pd.DataFrame(dev_rows).T
dev.index.name = "faultNumber"
gc.collect()
print("Done. f_train never held in memory.")

In [ ]:
# Load only the columns actually used in sections 4 and 8
_ff_cols = list(dict.fromkeys(["simulationRun", "sample"] + COMPARE_VARS + TRAJ_VARS))
print("Loading ff_test (column-projected)...")
_ff_test = pd.read_parquet(FF_TEST, columns=_ff_cols)
print(f"  {_ff_test.shape[0]:>8,} rows × {_ff_test.shape[1]} cols  "
      f"({_ff_test.memory_usage(deep=True).sum() / 1e6:.0f} MB)")

ff_run  = _ff_test[_ff_test["simulationRun"] == 1].sort_values("sample").copy()
ff_traj = _ff_test.groupby("sample")[TRAJ_VARS].mean()
del _ff_test
gc.collect()
print("ff_test freed.")

In [ ]:
# Load only the 5 fault types we actually plot, and only the columns we need.
# Reduces f_test from ~2.1 GB to ~90 MB in memory.
_f_cols   = list(dict.fromkeys(["faultNumber", "simulationRun", "sample"] + COMPARE_VARS + TRAJ_VARS))
_f_faults = list(set(FAULT_NUMBERS) | set(TRAJ_FAULTS))
print("Loading f_test (fault-filtered, column-projected)...")
_f_test = pd.read_parquet(F_TEST, columns=_f_cols, filters=[("faultNumber", "in", _f_faults)])
print(f"  {_f_test.shape[0]:>8,} rows × {_f_test.shape[1]} cols  "
      f"({_f_test.memory_usage(deep=True).sum() / 1e6:.0f} MB)")

faulty_runs = {
    fnum: _f_test[(_f_test["faultNumber"] == fnum) & (_f_test["simulationRun"] == 1)]
              .sort_values("sample").copy()
    for fnum in FAULT_NUMBERS
}
f_trajs = {
    fnum: _f_test[_f_test["faultNumber"] == fnum].groupby("sample")[TRAJ_VARS].mean()
    for fnum in TRAJ_FAULTS
}
del _f_test
gc.collect()
print("f_test freed.")

## 2. Basic Exploration

In [ ]:
# Schema and data types
print("Columns:", ff_train.columns.tolist())
print("\nData types:")
print(ff_train.dtypes)

In [ ]:
# Missing values
print("Missing values (fault-free training):")
nulls = ff_train.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else "  None")

In [ ]:
# Descriptive statistics for fault-free training features
ff_train[FEATURES].describe().T.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(fault_counts["faultNumber"].astype(int), fault_counts["runs"], color="steelblue")
ax.set_xlabel("Fault Number (IDV)")
ax.set_ylabel("Simulation Runs")
ax.set_title("Number of Simulation Runs per Fault Type (Faulty Training)")
ax.set_xticks(fault_counts["faultNumber"].astype(int))
plt.tight_layout()
plt.show()

## 3. Time Series — Normal Operation

One simulation run from the fault-free training set, showing all 52 process variables.

In [ ]:
# Pick one run and plot all features
run_id = 1
run = ff_train[ff_train["simulationRun"] == run_id].sort_values("sample")
time_h = run["sample"] * 3 / 60  # convert samples to hours

n_xmeas, n_xmv = len(XMEAS), len(XMV)
fig, axes = plt.subplots(n_xmeas, 1, figsize=(14, n_xmeas * 0.9), sharex=True)
fig.suptitle(f"Fault-Free Training — Run {run_id} — xmeas_1 to xmeas_41", y=1.001, fontsize=11)
for ax, col in zip(axes, XMEAS):
    ax.plot(time_h, run[col], lw=0.7, color="royalblue")
    ax.set_ylabel(col, fontsize=6, rotation=0, labelpad=40, va="center")
    ax.tick_params(axis="y", labelsize=6)
axes[-1].set_xlabel("Time (hours)")
plt.tight_layout()
plt.show()

In [ ]:
# Manipulated variables (xmv_1–11)
fig, axes = plt.subplots(n_xmv, 1, figsize=(14, n_xmv * 1.1), sharex=True)
fig.suptitle(f"Fault-Free Training — Run {run_id} — xmv_1 to xmv_11", fontsize=11)
for ax, col in zip(axes, XMV):
    ax.plot(time_h, run[col], lw=0.8, color="darkorange")
    ax.set_ylabel(col, fontsize=7, rotation=0, labelpad=40, va="center")
    ax.tick_params(axis="y", labelsize=7)
axes[-1].set_xlabel("Time (hours)")
plt.tight_layout()
plt.show()

## 4. Fault vs. Normal — Time Series Comparison

Compare the same 6 key variables across fault-free and a few fault types (IDV 1, 4, 5).  
Faults are injected at sample 160 (≈ 8 hours).

In [ ]:
fig, axes = plt.subplots(len(COMPARE_VARS), 1, figsize=(14, len(COMPARE_VARS) * 2.2), sharex=True)
colors = {"normal": "royalblue", 1: "tomato", 4: "seagreen", 5: "darkorchid"}

for ax, var in zip(axes, COMPARE_VARS):
    t = ff_run["sample"] * 3 / 60
    ax.plot(t, ff_run[var], lw=1, label="Normal", color=colors["normal"], alpha=0.85)
    for fnum, frun in faulty_runs.items():
        tf = frun["sample"] * 3 / 60
        ax.plot(tf, frun[var], lw=1, label=f"IDV {fnum}", color=colors[fnum], alpha=0.85)
    ax.axvline(FAULT_INJECTION_SAMPLE * 3 / 60, color="black", lw=1, ls="--", alpha=0.5)
    ax.set_ylabel(var, fontsize=9)
    ax.legend(fontsize=7, loc="upper right", ncol=4)

axes[-1].set_xlabel("Time (hours)")
fig.suptitle("Fault-Free vs. IDV 1/4/5 — Testing Set (Run 1)", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Feature Distributions — Normal vs. All Faults

Box plots and violin plots comparing the distribution of each feature group between normal and faulty operation (post-fault injection window only).

In [ ]:
ff_sample = ff_train[FEATURES].sample(5000, random_state=42).copy()
ff_sample["label"] = "Normal"
combined = pd.concat([ff_sample, faulty_post_sample], ignore_index=True)

# Box plots — xmeas group (first 22 continuous measurements)
plot_vars = XMEAS[:22]
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
axes = axes.flatten()
for ax, var in zip(axes, plot_vars):
    sns.boxplot(data=combined, x="label", y=var, ax=ax,
                palette={"Normal": "royalblue", "Faulty": "tomato"}, width=0.5, linewidth=0.8)
    ax.set_title(var, fontsize=8)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", labelsize=7)
for ax in axes[len(plot_vars):]:
    ax.set_visible(False)
fig.suptitle("Distribution: Normal vs. Faulty — xmeas_1 to xmeas_22", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots — manipulated variables (xmv_1–11)
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
axes = axes.flatten()
for ax, var in zip(axes, XMV):
    sns.violinplot(data=combined, x="label", y=var, ax=ax,
                   palette={"Normal": "royalblue", "Faulty": "tomato"}, linewidth=0.8)
    ax.set_title(var, fontsize=8)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", labelsize=7)
for ax in axes[len(XMV):]:
    ax.set_visible(False)
fig.suptitle("Distribution: Normal vs. Faulty — xmv_1 to xmv_11 (Manipulated Variables)", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap — Normal Operation

In [ ]:
corr = ff_train[FEATURES].sample(10000, random_state=42).corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.2, annot=False, square=True,
    cbar_kws={"shrink": 0.7, "label": "Pearson r"},
)
ax.set_title("Feature Correlation — Fault-Free Training (10k sample)", fontsize=12)
plt.tight_layout()
plt.show()

## 7. Per-Fault Mean Deviation from Normal

For each fault type, compute the mean absolute deviation of each feature from the fault-free mean.  
This shows which variables are most discriminative for each fault.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(
    dev.T, ax=ax,
    cmap="YlOrRd", linewidths=0.3,
    cbar_kws={"label": "Mean |z-score| (post-injection)"},
    yticklabels=True,
)
ax.set_xlabel("Fault Number (IDV)")
ax.set_ylabel("Feature")
ax.set_title("Feature Sensitivity per Fault Type (normalized deviation from normal mean)")
ax.tick_params(axis="y", labelsize=6)
plt.tight_layout()
plt.show()

## 8. Average Fault Trajectory

Show how the process mean evolves over time for normal vs. selected faults, averaged across all runs.  
The dashed vertical line marks the fault injection point.

In [ ]:
fig, axes = plt.subplots(len(TRAJ_VARS), 1, figsize=(13, len(TRAJ_VARS) * 2.5), sharex=True)
palette = plt.cm.tab10.colors

for ax, var in zip(axes, TRAJ_VARS):
    t = ff_traj.index * 3 / 60
    ax.plot(t, ff_traj[var], lw=1.5, color="royalblue", label="Normal", zorder=3)
    for i, fnum in enumerate(TRAJ_FAULTS):
        tf = f_trajs[fnum].index * 3 / 60
        ax.plot(tf, f_trajs[fnum][var], lw=1, color=palette[i + 1], label=f"IDV {fnum}", alpha=0.85)
    ax.axvline(FAULT_INJECTION_SAMPLE * 3 / 60, color="black", lw=1, ls="--", alpha=0.4)
    ax.set_ylabel(var, fontsize=9)
    ax.legend(fontsize=7, loc="upper right", ncol=5)

axes[-1].set_xlabel("Time (hours)")
fig.suptitle("Mean Trajectory — Normal vs. Selected Faults (Testing Set)", fontsize=12)
plt.tight_layout()
plt.show()